# Road Segmentation with U-Net (PyTorch)

**Pipeline:** load data → look at samples → build U-Net → train with live plots → evaluate → visualize predictions

Expected folders (image and mask share the same filename stem):
```
data/
  images/  img001.png ...
  masks/   img001.png ...
```
Masks are binary (road = white / non-zero, background = black).

## 1. Imports

In [2]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import clear_output

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

## 2. Config
Change everything you might want to tweak here.

In [3]:
IMAGE_DIR  = "data/images"
MASK_DIR   = "data/masks"

IMG_SIZE   = (256, 256)     # (height, width) - use (128, 128) for faster CPU training
BATCH_SIZE = 8
EPOCHS     = 30
LR         = 1e-3
VAL_SPLIT  = 0.2
SEED       = 42
FEATURES   = (64, 128, 256, 512)   # U-Net widths - use (16, 32, 64, 128) for a lighter model

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

Using device: cpu


In [ ]:
def create_dummy_data(num_samples=10):
    """Creates a small synthetic dataset to verify the pipeline works."""
    os.makedirs(IMAGE_DIR, exist_ok=True)
    os.makedirs(MASK_DIR, exist_ok=True)
    
    print(f"Generating {num_samples} dummy samples...")
    for i in range(num_samples):
        # Create a random noise image
        img = np.random.randint(0, 256, (256, 256, 3), dtype=np.uint8)
        # Create a simple rectangle as a road mask
        mask = np.zeros((256, 256), dtype=np.uint8)
        mask[100:150, 20:230] = 255 
        
        Image.fromarray(img).save(os.path.join(IMAGE_DIR, f"dummy_{i}.png"))
        Image.fromarray(mask).save(os.path.join(MASK_DIR, f"dummy_{i}.png"))
    print("Dummy data created successfully!")

# Run this if you don't have your own data yet
if not os.path.exists(IMAGE_DIR):
    create_dummy_data()

## 3. Dataset
Resizes, normalizes, and (for training only) randomly flips image and mask together.

In [ ]:
class RoadDataset(Dataset):
    def __init__(self, image_dir, mask_dir, img_size, augment=False):
        self.image_dir, self.mask_dir = image_dir, mask_dir
        self.img_size, self.augment = img_size, augment

        if not os.path.exists(image_dir) or not os.path.exists(mask_dir):
            print(f"Warning: Data directories not found ({image_dir} or {mask_dir}). Dataset will be empty.")
            self.files = []
            self.mask_lookup = {}
            return

        exts = (".png", ".jpg", ".jpeg", ".tif", ".bmp")
        self.files = sorted(f for f in os.listdir(image_dir) if f.lower().endswith(exts))
        self.mask_lookup = {os.path.splitext(f)[0]: f for f in os.listdir(mask_dir)}

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        name = self.files[idx]
        stem = os.path.splitext(name)[0]

        image = Image.open(os.path.join(self.image_dir, name)).convert("RGB")
        mask  = Image.open(os.path.join(self.mask_dir, self.mask_lookup[stem])).convert("L")

        h, w = self.img_size
        image = image.resize((w, h), Image.BILINEAR)
        mask  = mask.resize((w, h), Image.NEAREST)      # NEAREST keeps mask values clean

        if self.augment:
            if random.random() < 0.5:
                image, mask = image.transpose(Image.FLIP_LEFT_RIGHT), mask.transpose(Image.FLIP_LEFT_RIGHT)
            if random.random() < 0.5:
                image, mask = image.transpose(Image.FLIP_TOP_BOTTOM), mask.transpose(Image.FLIP_TOP_BOTTOM)

        img = torch.from_numpy(np.array(image)).permute(2, 0, 1).float() / 255.0
        img = (img - MEAN) / STD

        mask = (np.array(mask) > 0).astype(np.float32)   # works for 0/1 and 0/255 masks
        mask = torch.from_numpy(mask).unsqueeze(0)       # shape [1, H, W]
        return img, mask

## 4. Train / validation split and a first look at the data

In [5]:
# Two dataset objects over the same files: augmentation only for training
train_full = RoadDataset(IMAGE_DIR, MASK_DIR, IMG_SIZE, augment=True)
val_full   = RoadDataset(IMAGE_DIR, MASK_DIR, IMG_SIZE, augment=False)

indices = torch.randperm(len(train_full), generator=torch.Generator().manual_seed(SEED)).tolist()
n_val = int(len(indices) * VAL_SPLIT)
val_ds   = Subset(val_full,   indices[:n_val])
train_ds = Subset(train_full, indices[n_val:])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=(DEVICE == "cuda"))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=(DEVICE == "cuda"))

print(f"Total images: {len(train_full)} | train: {len(train_ds)} | val: {len(val_ds)}")

FileNotFoundError: [Errno 2] No such file or directory: 'data/images'

In [ ]:
def to_display(img_tensor):
    """Undo normalization so a [3,H,W] tensor can be shown with matplotlib."""
    return (img_tensor * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy()

def show_samples(dataset, n=4):
    fig, axes = plt.subplots(2, n, figsize=(3.5 * n, 7))
    for i in range(n):
        img, mask = dataset[i]
        axes[0, i].imshow(to_display(img));            axes[0, i].set_title(f"Image {i}")
        axes[1, i].imshow(mask[0], cmap="gray");       axes[1, i].set_title("Mask")
        axes[0, i].axis("off"); axes[1, i].axis("off")
    plt.tight_layout(); plt.show()

show_samples(val_ds, n=4)

# Sanity check: what fraction of pixels is road?
_, m = val_ds[0]
print(f"Road pixels in first sample: {m.mean().item() * 100:.1f}%  (0% means your masks may be empty!)")

## 5. Model: U-Net

In [ ]:
class DoubleConv(nn.Sequential):
    def __init__(self, in_ch, out_ch):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=(64, 128, 256, 512)):
        super().__init__()
        self.pool = nn.MaxPool2d(2)

        # Encoder (downsampling path)
        self.downs = nn.ModuleList()
        ch = in_channels
        for f in features:
            self.downs.append(DoubleConv(ch, f))
            ch = f

        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)

        # Decoder (upsampling path): each stage = upsample + DoubleConv
        self.up_convs = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        for f in reversed(features):
            self.up_convs.append(nn.ConvTranspose2d(f * 2, f, kernel_size=2, stride=2))
            self.dec_convs.append(DoubleConv(f * 2, f))

        self.head = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)

        for up, dec, skip in zip(self.up_convs, self.dec_convs, reversed(skips)):
            x = up(x)
            if x.shape[-2:] != skip.shape[-2:]:
                x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = dec(torch.cat([skip, x], dim=1))

        return self.head(x)    # raw logits (sigmoid is applied in loss / prediction)

model = UNet(features=FEATURES).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"U-Net ready: {n_params / 1e6:.1f}M parameters")

# Quick shape check
with torch.no_grad():
    dummy = torch.randn(1, 3, *IMG_SIZE).to(DEVICE)
    print("Input:", tuple(dummy.shape), "-> Output:", tuple(model(dummy).shape))

## 6. Loss and metrics
**Loss** = BCE + Dice. **IoU** and **Dice score** are reported each epoch.

In [ ]:
class DiceBCELoss(nn.Module):
    def forward(self, logits, targets, eps=1e-6):
        bce = F.binary_cross_entropy_with_logits(logits, targets)
        probs = torch.sigmoid(logits)
        inter = (probs * targets).sum(dim=(1, 2, 3))
        total = probs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
        dice_loss = 1 - ((2 * inter + eps) / (total + eps)).mean()
        return bce + dice_loss

@torch.no_grad()
def iou_and_dice(logits, targets, eps=1e-6):
    preds = (torch.sigmoid(logits) > 0.5).float()
    inter = (preds * targets).sum()
    union = preds.sum() + targets.sum() - inter
    iou  = (inter + eps) / (union + eps)
    dice = (2 * inter + eps) / (preds.sum() + targets.sum() + eps)
    return iou.item(), dice.item()

## 7. Training helpers (with live plot)

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    """One pass over the loader. Trains if an optimizer is given, otherwise just evaluates."""
    training = optimizer is not None
    model.train(training)
    loss_sum = iou_sum = dice_sum = 0.0

    with torch.set_grad_enabled(training):
        for imgs, masks in loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            logits = model(imgs)
            loss = criterion(logits, masks)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            iou, dice = iou_and_dice(logits, masks)
            loss_sum += loss.item(); iou_sum += iou; dice_sum += dice

    n = len(loader)
    return loss_sum / n, iou_sum / n, dice_sum / n


def plot_history(history):
    clear_output(wait=True)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(history["train_loss"], label="Train", color="tab:blue")
    ax1.plot(history["val_loss"],   label="Val",   color="tab:red")
    ax1.set_title("Loss"); ax1.set_xlabel("Epoch"); ax1.legend(); ax1.grid(alpha=0.3)

    ax2.plot(history["train_iou"], label="Train IoU", color="tab:blue")
    ax2.plot(history["val_iou"],   label="Val IoU",   color="tab:red")
    ax2.set_title("IoU"); ax2.set_xlabel("Epoch"); ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout(); plt.show()

## 8. Train

In [ ]:
criterion = DiceBCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

history = {k: [] for k in ["train_loss", "val_loss", "train_iou", "val_iou"]}
best_iou = 0.0

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_iou, _ = run_epoch(model, train_loader, criterion, optimizer)
    va_loss, va_iou, _ = run_epoch(model, val_loader, criterion)
    scheduler.step(va_loss)

    for k, v in zip(history, [tr_loss, va_loss, tr_iou, va_iou]):
        history[k].append(v)

    saved = ""
    if va_iou > best_iou:
        best_iou = va_iou
        torch.save(model.state_dict(), "best_model.pth")
        saved = "  <- saved best"

    plot_history(history)
    print(f"Epoch {epoch:02d}/{EPOCHS} | train loss {tr_loss:.4f}, IoU {tr_iou:.4f} "
          f"| val loss {va_loss:.4f}, IoU {va_iou:.4f}{saved}")

print(f"\nBest validation IoU: {best_iou:.4f}")

## 9. Evaluate the best model

In [ ]:
model.load_state_dict(torch.load("best_model.pth", map_location=DEVICE))
val_loss, val_iou, val_dice = run_epoch(model, val_loader, criterion)
print(f"Validation  ->  loss: {val_loss:.4f} | IoU: {val_iou:.4f} | Dice: {val_dice:.4f}")

## 10. Visualize predictions
Each row: **image | ground truth | prediction | overlay** (green = correct road, red = false road, blue = missed road).

In [ ]:
@torch.no_grad()
def predict_batch(images):
    model.eval()
    return (torch.sigmoid(model(images.to(DEVICE))) > 0.5).float().cpu()

def show_predictions(dataset, n=4, start=0):
    fig, axes = plt.subplots(n, 4, figsize=(14, 3.4 * n))
    titles = ["Image", "Ground truth", "Prediction", "Overlay"]

    for row in range(n):
        img, gt = dataset[start + row]
        pred = predict_batch(img.unsqueeze(0))[0]

        gt_np, pred_np = gt[0].numpy() > 0.5, pred[0].numpy() > 0.5
        overlay = np.zeros((*gt_np.shape, 3))
        overlay[gt_np & pred_np]  = [0, 1, 0]   # correct   (green)
        overlay[~gt_np & pred_np] = [1, 0, 0]   # false pos (red)
        overlay[gt_np & ~pred_np] = [0, 0, 1]   # missed    (blue)

        iou, _ = iou_and_dice(torch.from_numpy(pred_np.astype(np.float32)), torch.from_numpy(gt_np.astype(np.float32)))

        panels = [to_display(img), gt_np, pred_np, overlay]
        for col, (ax, panel) in enumerate(zip(axes[row], panels)):
            ax.imshow(panel, cmap="gray" if col in (1, 2) else None)
            ax.axis("off")
            ax.set_title(titles[col] + (f"  (IoU {iou:.2f})" if col == 3 else ""))

    plt.tight_layout(); plt.show()

show_predictions(val_ds, n=4)

## 11. Predict on any new image

In [ ]:
def predict_image(path, show=True):
    """Run the model on one image file. Returns the binary mask at the original image size."""
    original = Image.open(path).convert("RGB")
    h, w = IMG_SIZE
    x = torch.from_numpy(np.array(original.resize((w, h), Image.BILINEAR))).permute(2, 0, 1).float() / 255.0
    x = ((x - MEAN) / STD).unsqueeze(0)

    mask = predict_batch(x)[0, 0].numpy().astype(np.uint8)
    mask_full = np.array(Image.fromarray(mask * 255).resize(original.size, Image.NEAREST)) > 0

    if show:
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(original);                       axes[0].set_title("Image")
        axes[1].imshow(mask_full, cmap="gray");         axes[1].set_title("Predicted mask")
        axes[2].imshow(original); axes[2].imshow(mask_full, alpha=0.4, cmap="Reds"); axes[2].set_title("Overlay")
        for ax in axes: ax.axis("off")
        plt.tight_layout(); plt.show()
    return mask_full

# Example:
# predict_image("data/images/img001.png")